# Reproducing — 12-Lead ECG Arrhythmia Classification (CPSC-2018)

This notebook reproduces the results of
**[ecg-arrhythmia-classification](https://github.com/Kevindic0214/ecg-arrhythmia-classification)**
end-to-end on a free Colab **GPU**:

1. Download the **real CPSC-2018** data from PhysioNet (the Challenge-2020 repackaging).
2. Preprocess it with the repo's **own** pipeline (`src/preprocessing.py`).
3. Train the repo's **CNN + Bi-GRU + Attention** model (`src/model.py`, 934,993 params)
   with the full protocol (Adam 1e-4, dropout 0.3, LR halved every 10 epochs,
   early stopping, cost-sensitive loss).
4. Evaluate: AUC / macro-F1 / accuracy + a normalized confusion matrix.

> **First:** `Runtime → Change runtime type → GPU (T4)`.
> Full training is roughly **20–40 min** on a T4 (less with early stopping).
> To do a fast trial first, set `FULL_DATASET = False` in the download cell.

> **Note on labels.** PhysioNet stores CPSC-2018 with SNOMED-CT `Dx` codes in each
> `.hea` header; we map them back to the original 9 CPSC classes.


In [ ]:
# 0. Check the GPU is on
!nvidia-smi -L || echo "No GPU — set Runtime > Change runtime type > GPU (T4)"


In [ ]:
# 1. Get the repo + dependencies
!git clone -q https://github.com/Kevindic0214/ecg-arrhythmia-classification.git
!pip install -q PyWavelets            # torch / numpy / scipy / sklearn / matplotlib ship with Colab
import sys
sys.path.insert(0, "/content/ecg-arrhythmia-classification/src")
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


In [ ]:
# 2. Download the real CPSC-2018 records from PhysioNet
#    Records are A0001..A6877, grouped 1000 per folder (g1..g7).
import os, urllib.request, concurrent.futures as cf

BASE = "https://physionet.org/files/challenge-2020/1.0.2/training/cpsc_2018"
DATA = "/content/data"; os.makedirs(DATA, exist_ok=True)

FULL_DATASET = True          # <- set False for a quick trial on N_TRIAL records
N_TRIAL = 800
N = 6877 if FULL_DATASET else N_TRIAL

def url(i, ext):
    g = (i - 1) // 1000 + 1
    return f"{BASE}/g{g}/A{i:04d}{ext}"

def fetch(i):
    rec = f"A{i:04d}"; out = []
    for ext in (".hea", ".mat"):
        dst = os.path.join(DATA, rec + ext)
        if not os.path.exists(dst):
            try: urllib.request.urlretrieve(url(i, ext), dst)
            except Exception: return None
        out.append(dst)
    return rec

print(f"Downloading {N} records ...")
with cf.ThreadPoolExecutor(max_workers=32) as ex:
    got = [r for r in ex.map(fetch, range(1, N + 1)) if r]
print("downloaded", len(got), "records")


In [ ]:
# 3. Parse labels (SNOMED -> 9 CPSC classes) and preprocess with the repo's pipeline
import numpy as np
from scipy.io import loadmat
from preprocessing import preprocess_recording
from dataset import CLASS_NAMES

# CPSC order: Normal, AF, I-AVB, LBBB, RBBB, PAC, PVC, STD, STE
SNOMED_TO_IDX = {
    "426783006": 0,
    "164889003": 1,
    "270492004": 2,
    "164909002": 3,
    "59118001": 4,  "713427006": 4,
    "284470004": 5, "63593006": 5,
    "427172004": 6, "164884008": 6,
    "429622005": 7,
    "164931005": 8,
}

def label_of(rec):
    with open(os.path.join(DATA, rec + ".hea")) as f:
        for line in f:
            if line.startswith("# Dx:"):
                for c in (x.strip() for x in line.split(":", 1)[1].split(",")):
                    if c in SNOMED_TO_IDX:
                        return SNOMED_TO_IDX[c]
    return None

X, y = [], []
for k, rec in enumerate(got):
    idx = label_of(rec)
    if idx is None:
        continue
    sig = loadmat(os.path.join(DATA, rec + ".mat"))["val"].astype(np.float64) / 1000.0
    for w in preprocess_recording(sig):            # (n_windows, 12, 5000)
        X.append(w.astype(np.float32)); y.append(idx)
    if (k + 1) % 1000 == 0:
        print(f"  preprocessed {k+1}/{len(got)}")

X = np.stack(X); y = np.array(y, np.int64)
print("windows:", X.shape, "labels:", y.shape)
for c in range(9):
    print(f"  {CLASS_NAMES[c]:7s}: {(y==c).sum()}")


In [ ]:
# 4. Train — full protocol on GPU (mirrors src/train.py)
import copy, torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from dataset import ECGWindowDataset, class_weights
from model import ECGArrhythmiaNet

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
tr = DataLoader(ECGWindowDataset(Xtr, ytr), batch_size=32, shuffle=True)
va = DataLoader(ECGWindowDataset(Xva, yva), batch_size=64)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ECGArrhythmiaNet().to(dev)
crit = nn.CrossEntropyLoss(weight=class_weights(ytr).to(dev))   # cost-sensitive
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

EPOCHS, PATIENCE = 100, 10
hist = {"tr_loss": [], "va_loss": [], "va_acc": []}
best, best_state, wait = float("inf"), None, 0

def run(loader, train):
    model.train(train); tot = cor = n = 0.0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            logits = model(xb); loss = crit(logits, yb)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(yb); cor += (logits.argmax(1)==yb).sum().item(); n += len(yb)
    return tot/n, cor/n

for ep in range(1, EPOCHS+1):
    trl, _ = run(tr, True)
    val, vac = run(va, False)
    sched.step()
    hist["tr_loss"].append(trl); hist["va_loss"].append(val); hist["va_acc"].append(vac)
    print(f"[{ep:3d}] train_loss {trl:.4f} | val_loss {val:.4f} val_acc {vac:.4f}")
    if val < best:
        best, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {ep} (best val_loss {best:.4f})"); break
model.load_state_dict(best_state)


In [ ]:
# 5. Evaluate — AUC / macro-F1 / accuracy + confusion matrix
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

model.eval(); probs, tgts = [], []
with torch.no_grad():
    for xb, yb in va:
        probs.append(torch.softmax(model(xb.to(dev)), 1).cpu().numpy()); tgts.append(yb.numpy())
probs = np.concatenate(probs); tgts = np.concatenate(tgts); preds = probs.argmax(1)

acc = accuracy_score(tgts, preds)
f1  = f1_score(tgts, preds, average="macro")
try:
    auc = roc_auc_score(np.eye(9)[tgts], probs, average="macro", multi_class="ovr")
except ValueError:
    auc = float("nan")
print(f"Accuracy {acc:.4f} | macro-F1 {f1:.4f} | macro-AUC {auc:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(hist["tr_loss"], label="train loss"); ax[0].plot(hist["va_loss"], label="val loss")
ax[0].plot(hist["va_acc"], label="val acc"); ax[0].legend(); ax[0].set_xlabel("epoch"); ax[0].set_title("Training")
cm = confusion_matrix(tgts, preds, labels=list(range(9)), normalize="true")
im = ax[1].imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax[1].set_xticks(range(9)); ax[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax[1].set_yticks(range(9)); ax[1].set_yticklabels(CLASS_NAMES)
ax[1].set_title("Normalized confusion matrix"); fig.colorbar(im, ax=ax[1])
for i in range(9):
    for j in range(9):
        ax[1].text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                   color="white" if cm[i,j] > 0.5 else "black", fontsize=7)
plt.tight_layout(); plt.show()
